# NLP Practical Exam — Text Processing + Language Modeling (90 minutes)

**Instructions**
- Work in this notebook only.
- Write short, clear comments to justify *tool choices* (regex vs NLTK, etc.).
- Do **not** use external NLP libraries beyond **NLTK**, **NumPy**, **PyTorch** (PyTorch not needed here).
- Keep outputs readable (print key variables).

**Total: 10 points**


## Given text

```python
text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.")
```

> Treat the text as *synthetic exam data* (no fact-checking needed).


## Questions

1. **(1 pt)** Sentence splitting using **regex + NLTK**.
2. **(1 pt)** Regex normalization: acronyms, height meters→centimeters, money `$X.Y billion` → `x point y billion` (words).
3. **(1 pt)** Lowercase **except** proper nouns; join multiword proper nouns with underscore (e.g., `Sam Altman → Sam_Altman`). Keep acronyms uppercase.
4. **(1 pt)** Tokenize (tool of your choice).
5. **(1 pt)** Remove stopwords (tool of your choice); keep entity tokens.
6. **(1 pt)** Create bigrams with pure Python.
7. **(2 pt)** Build a bigram LM (MLE) and `predict_next(prev_word, top_k=3)`.

8. **(2 pt)** Implement a simple **BPE** on: `corpus = "low lower newest widest"` (≥5 merges or until no merges).
9. **(1 pt)** Compute Accuracy/Precision/Recall/F1 for an invented confusion matrix (explain with comments).


In [49]:
import re
import math
import nltk
from collections import Counter, defaultdict

# NLTK downloads (safe to run multiple times)
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. "
        "He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. "
        "A report valued the project at $3.2 billion.")

print(text)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.


## Q1

In [50]:
# Q1 (1 pt): Sentence splitting (regex + NLTK)
# - Use regex to protect acronyms like U.P.C. so they don't break sentence boundaries.
abbreviations = r"\b(?:[A-Z]\.){2,}"

def protect_acronym_dots(t):
    return re.sub(abbreviations, lambda m: m.group(0).replace(".", "<DOT>"), t)

def restore_acronym_dots(t):
    return t.replace("<DOT>", ".")

# - Then use nltk.sent_tokenize.
protected = protect_acronym_dots(text)
sent = sent_tokenize(protected)
sentences = [restore_acronym_dots(s) for s in sent]

print(sentences)

['In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.', 'He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.']


## Q2

In [51]:
# Q2 (1 pt): Regex normalization
# Convert:
#  - U.P.C. -> UPC, U.N.E.S.C.O. -> UNESCO (general rule: remove dots in acronyms)
def remove_dots_from_acronyms(t):
    return re.sub(abbreviations, lambda m: m.group(0).replace(".", ""), t)
# print(remove_dots_from_acronyms(text))

#  - 1.86m -> 186 centimeters (general: X.YZm -> int(round(float(X.YZ)*100)) centimeters)
def use_centimeters(t):
    return re.sub(r'(\d+\.\d+)m', lambda x: f"{int(round(float(x.group(1)) * 100))} centimeters", t)
# print(use_centimeters(text))

#  - $3.2 billion -> three point two billion  (digits 0-9 are enough)
from num2words import num2words

def decimal_to_words(t):
    number = t.group(1) 
    integer_part, decimal_part = number.split(".")
    return f"{num2words(int(integer_part))} point {num2words(int(decimal_part))} billion"

text_dw = re.sub(r'\$(\d+\.\d+)\sbillion', decimal_to_words, text)
# print(text_dw)

# Return: text_norm

text_norm = remove_dots_from_acronyms(text)
text_norm = use_centimeters(text_norm)
text_norm = re.sub(r'\$(\d+\.\d+)\sbillion', decimal_to_words, text_norm)

print(text_norm)

In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion.


## Q3

In [59]:
# Q3 (1 pt): Lowercase except proper nouns + underscore multiword proper nouns
# Requirements:
# - Convert to lowercase except:
#   - Acronyms (ALL CAPS) stay uppercase (e.g., UNESCO, UPC, CEO)
#   - MixedCase tokens stay as-is (e.g., OpenAI)
#   - Multiword proper nouns joined with underscore (Sam Altman -> Sam_Altman) and preserved
#
# Return: text_case
from nltk.tokenize.treebank import TreebankWordDetokenizer
def lowercase_text(t):
    multiword_proper = ["Sam Altman"]
    single_proper = {"Barcelona"}
    
    for name in multiword_proper:
        t = re.sub(r"\b" + re.escape(name) + r"\b", name.replace(" ", "_"), t)
    
    tokens = word_tokenize(t)
    processed = []
    for tok in tokens:
        if re.fullmatch(r"[^\w\s]", tok) or re.fullmatch(r"\d+(?:\.\d+)?", tok):
            processed.append(tok)
        elif "_" in tok or tok in single_proper:
            processed.append(tok)
        elif tok.isupper() and len(tok) > 1:
            processed.append(tok)
        # MixedCase means internal uppercase, not just initial capital
        elif re.search(r"[A-Z]", tok[1:]) and re.search(r"[a-z]", tok):
            processed.append(tok)
        else:
            processed.append(tok.lower())
    
    detok = TreebankWordDetokenizer().detokenize(processed)
    return re.sub(r"\s+([.,!?;:])", r"\1", detok)

text_case = lowercase_text(text_norm)

print(text_case)

in mid-February 2026, the CEO of OpenAI, Sam_Altman, visited Barcelona. he is 186 centimeters tall and met with researchers from UPC and UNESCO a report valued the project at three point two billion.


## Q4

In [60]:
# Q4 (1 pt): Tokenization
# Use a tokenizer of your choice (e.g., nltk.word_tokenize).
# Return: tokens (list)

tokens = nltk.word_tokenize(text_case)

print(tokens)


['in', 'mid-February', '2026', ',', 'the', 'CEO', 'of', 'OpenAI', ',', 'Sam_Altman', ',', 'visited', 'Barcelona', '.', 'he', 'is', '186', 'centimeters', 'tall', 'and', 'met', 'with', 'researchers', 'from', 'UPC', 'and', 'UNESCO', 'a', 'report', 'valued', 'the', 'project', 'at', 'three', 'point', 'two', 'billion', '.']


## Q5

In [54]:
# Q5 (1 pt): Stopword removal
# - Remove English stopwords
# - Do NOT remove entity tokens like OpenAI, Sam_Altman, Barcelona, UNESCO, UPC
# Return: tokens_nostop

tokens_nostop = None

# print(tokens_nostop)


## Q6

In [55]:
# Q6 (1 pt): Bigrams with pure Python (no NLTK bigrams helper)
# Return: bigrams = [(w1, w2), ...]

bigrams = None

# print(bigrams)


## Q7

In [56]:
# Q7 (2 pt): Bigram Language Model + next-word prediction
# Build:
# - bigram_counts[(w1,w2)]
# - context_counts[w1]
# - model[w1][w2] = P(w2|w1) = count(w1,w2)/count(w1)
#
# Then implement:
# def predict_next(prev_word, model, top_k=3): -> list[(next_word, prob)] sorted

bigram_counts = None
context_counts = None
model = None

def predict_next(prev_word, model, top_k=3):
    # TODO
    return None

# Example:
# print(predict_next("OpenAI", model, top_k=3))


## Q8

In [57]:
# Q8 (2 pt): Simple BPE (Byte Pair Encoding) on a tiny corpus
corpus = "low lower newest widest"

# Requirements:
# - Represent each word as characters + </w>
# - Compute pair frequencies (weighted by word frequency)
# - Merge most frequent pair
# - Do at least 5 merges (or stop if no pairs)
#
# Deliver:
# - merges: list of merges in order
# - final segmented version of each word

merges = None

# TODO: implement BPE helper functions:
# - get_vocab_from_corpus
# - get_pair_frequencies
# - merge_pair_in_vocab

# print(merges)


## Q9

In [58]:
# Q9 (1 pt): Metrics — Accuracy, Precision, Recall, F1
# Invent a confusion matrix (TP, FP, FN, TN) and compute metrics.
# Explain each formula briefly in comments.

TP = None
FP = None
FN = None
TN = None

accuracy = None
precision = None
recall = None
f1 = None

# print(accuracy, precision, recall, f1)
